Disclaimer
----------
Examples in this seminar run using VPython. Currently, Google Colab and JupyterLab do not support VPython. We recommend using the classic Jupyter Notebook with conda environments.

To set up a conda environment:
```
conda create -n seminar_env python=3.10
conda activate seminar_env
pip install vpython
pip install numpy scipy pyquaternion
pip install notebook   # for classic notebook
```

Try your VPython installation:

In [ ]:
import vpython as vp
scene = vp.canvas(title='sphere')
vp.sphere()

If nothing happens, standalone Python scripts with examples are provided:

```
python simple_sim_example_particles.py masses
python simple_sim_example_particles.py hanging
python simple_sim_example_particles.py wave

python simple_sim_example_rigid_bodies.py bodies_spring
python simple_sim_example_rigid_bodies.py hanging_body

python simple_sim_example_constraints.py
```

# Part 1. Particles Simulation

## 1.1. Particle System
### 1.1.1 Newton's Law

Dynamics of a particle (a geometric point with a positive scalar value $m$): 

$$\ddot{\bf{x}}=\frac{1}{m}\mathbf{f}$$

where $\mathbf{x}=\begin{pmatrix} x \\ y \\ z \end{pmatrix}$ is the radius vector of the particle, $\ddot{\mathbf{x}}=\frac{d^2\mathbf{x}}{dt^2}$ is the acceleration of the point, $\mathbf{f}$ is the cumulative force applied to the particle.

To handle a second-order ODE, we convert it to a first-order one by introducing an extra variable $\bf{v}=\dot{\bf{x}}$ (velocity), which forms the canonical form $\dot{\mathbf{x}}(t)=f(\mathbf{x}, t)$.

State of the particle $X$:

$$X=\begin{pmatrix}\mathbf{x} \\ \mathbf{v}\end{pmatrix}$$

However, we will use momentum instead of velocity:

$$X=\begin{pmatrix}\mathbf{x} \\ \mathbf{p}\end{pmatrix}$$

Motion equation:

$$\dot{X}=\frac{d}{dt}X$$

Where the right-hand side $\frac{d}{dt}X$ equals, according to Newton's Second Law:

$$\frac{d}{dt}X=\begin{pmatrix} \mathbf{v} \\ \mathbf{f} \end{pmatrix}$$

<!-- $$\begin{pmatrix} \dot{\mathbf{x}} \\ \dot{\mathbf{v}} \end{pmatrix} = \begin{pmatrix} \mathbf{v} \\ \frac{1}{m}\mathbf{f} \end{pmatrix}$$ -->

This system of ODEs can be solved numerically for $X$. The initial state of the particle is specified using the initial position and velocity $X_0=\begin{pmatrix} \mathbf{x}_0 \\ \mathbf{p}_0 \end{pmatrix}$. In this formulation, our task is referred to as the **initial value problem**.

<!-- $$\begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}$$ -->

### 1.1.2 Particle System

For $N$ points, the state of the system is described by $6N$-dimensional vectors: $$X=\begin{pmatrix} \bf{x}_1 \\ \bf{p}_1 \\ \bf{x}_2 \\ \bf{p}_2 \\ ... \\ \bf{p}_N \\ \bf{p}_N \end{pmatrix}$$.

The right-hand side of the motion equation for the system looks as follows:

$$\frac{d}{dt}X=\begin{pmatrix} \bf{v}_1 \\ \bf{f}_1 \\  \bf{v}_2 \\ \bf{f}_2 \\ ... \\ \bf{v}_N \\ \bf{f}_N \end{pmatrix}$$

Conceptually, the whole system may be regarded as a point moving through $6N$-dimensional space.

## 1.2 Particle Simulation

Simulation involves two parts: particles themselves and entities that apply forces to particles. Particles have mass, position, and velocity, and are subjected to forces, leading to an obvious structure definition: 

```
@dataclass
class Particle:
    m: float                                     # mass m
    position: Vec3                               # position vector x
    linear_momentum: Vec3                        # momentum vector p
    force_accumulator: vec3                      # f

    def get_state(self): ...                     # return concetenated pair (x, p) 
    def set_state(self, new_state: np.ndarray):  # set new x and p
    ...
```

There will be extra fields describing appearance and other properties, which we will cover later.
The important thing is to add the function `get_dx_dt` to obtain the right-hand side of the motion equations $\begin{pmatrix} \bf{v} \\ \bf{f} \end{pmatrix}$ for the particle:

```
    ...
    def get_dx_dt(self): 
        return dict(
            x_dot=self.linear_momentum / self.m, # velocity v
            p_dot=self.force_accumulator         # applied forces
        )
    ...
```

The class `PhysicsEngine` performs the whole physics step and manages all parts of the simulator. It contains:
* list of all particles
* forces that exist in the system
* collisions between particles - disregard in this part of this seminar
* constraints introduced to the system - disregard in this part of this seminar
* ODE solver which performs numerical integration of motion equation

```
@dataclass
class PhysicsEngins:
    bodies: List[Particle]
    forces: List[Force]                          # to be defined
    dynamics_solver: ODESolverBase               # to be defined

    def step(self, t_0, t_1):...                 # perform step

    # helper functions
    def get_system_state(self):...               # concatenate states from all particles in 3*N dimensional vector
    def set_system_state(self, new_state):...    # update state for each particle
    def system_dx_dt_func(self, ...)             # function that computes right-hand side for the system

    # other...
    ...
```


One step of the physics engine looks as follows (w/o collisions):

1. Zero out all force accumulators for each particle.
2. Compute forces applied to each particle.
3. Get system state (call `x = self.get_system_state()`).
4. Calculate right-hand side of motion equation (call `dx_dt = self.system_dx_dt_func(...)`.
5. Call ODE solver: `self.dynamics_solver.ode(x, dx_dt, t0, t_1, ...)`.

Note that it is more convenient to call `self.system_dx_dt_func` from within `self.dynamics_solver.ode()`, so we will skip step 4.

## 1.3 Forces

Forces change `force_accumulator` of particles to which they are applied.

For example, Hooke's force between two bodies `a` and `b` connected with spring can be calculated:

$$\mathbf{f}_a=-[k_s(|\mathbf{l}|-r)+k_d\frac{\dot{\mathbf{l}} \cdot \mathbf{l}}{|\mathbf{l}|}]\frac{\mathbf{l}}{|\mathbf{l}|}, \mathbf{f}_b = - \mathbf{f}_a $$

Gravity force:

$$\mathbf{f}=m\mathbf{g}$$

Viscous drag:

$$\mathbf{f}=-k_d\mathbf{v}$$

## 1.4 ODE solver

The simplest numerical method is called Euler’s method. Let our initial value for the system state be denoted by $X_0=X(t_0)$ and our estimate of $X$ at a later time $t_0+h$ be $X(t_0+h)$, where $h$ is a step-size parameter. Euler’s method simply computes $X(t_0+h)$ by taking a step in the derivative direction: $$X(t_0+h)=X_0+h\frac{d}{dt}X$$

Let's implement Euler's method.


In [ ]:
from typing import List, Optional, Union
from dataclasses import dataclass, field

import numpy as np
from pyquaternion.quaternion import Quaternion

import vpython as vp

from simple_sim_tutorial import *

In [ ]:
class EulerMethod(ODESolverBase):
    '''
    s_0 = (x_0, v_0)
    dx_dt_func = (f(x, v), g(x, v))

    Step:
    x_1 = x_0 + dt * f(x_0, v_0)
    v_1 = v_0 + dt * g(x_0, v_0)
    '''
    def ode(self, x_0, t_0, t_1, dx_dt_func):
        dt = t_1 - t_0 # stepsize
        f = dx_dt_func(x_0)
        x_1 = x_0['x'] + dt * f['x_dot']
        p_1 = x_0['p'] + dt * f['p_dot']
        return {
            'x': x_1,
            'p': p_1
        }

## 1.5 Implement Particle Simulator!

**It is more convenient to store the inverse mass $\frac{1}{m}$ instead of $m$** for each particle. If the inverse mass is zero, then the object is static (infinite mass). 

In [ ]:
@dataclass
class Particle(PhysObject):
    # constants
    inv_mass: float                            # 1 / m

    # state
    pos: Vec3 = Vec3(0, 0, 0)                  # x(t)
    linear_momentum: Vec3 = Vec3(0, 0, 0)      # P(t)

    # derived auxiliary variables
    @property
    def body_velocity(self):                   # P(t) / m
        return self.calc_velocity(self.linear_momentum)

    # we make separate function which uses only body constants (but no state variables) 
    # for calculation each auxiliary variables
    def calc_velocity(self, linear_momentum):
        return linear_momentum * self.inv_mass
        
    # computed quantites
    force_accumulator: Vec3 = Vec3(0, 0, 0)    # F(t) 

    def zero_forces(self):
        self.force_accumulator = Vec3(0, 0, 0)

    # return state
    def get_state(self):
        state = dict(
            x = self.pos,
            p = self.linear_momentum
        )
        return state

    # set state
    def set_state(self, new_state):
        assert len(new_state['x']) == self.state_x_size
        assert len(new_state['p']) == self.state_p_size
        self.pos = Vec3(new_state['x'])
        self.linear_momentum = Vec3(new_state['p'])

    # 3 for particles, 7 for rigid bodies
    @property
    def state_x_size(self):
        return len(self.get_state()['x'])

    # 3 for particles, 6 for rigid bodies
    @property
    def state_p_size(self):
        return len(self.get_state()['p'])

    @property
    def state_size(self):
        return self.state_x_size + self.state_p_size

    def dx_dt_func(self, x_state):
        x = x_state['x']
        linear_momentum = x_state['p']

        body_velocity = self.calc_velocity(linear_momentum)
        return {
            'x_dot': body_velocity,
            'p_dot': self.force_accumulator
        }
        

Let's implement some forces.

In [ ]:
class GravityForce(ForceBase):
    def __init__(self, g_vector:Vec3 = Vec3(0, -9.81, 0)):
        self.g_vector = g_vector

    def apply_force(self, bodies: List[Particle]):
        for body in bodies:
            if body.inv_mass > 0:
                body.force_accumulator += self.g_vector / body.inv_mass 

class HookeForce(ForceBase):
    def __init__(self, body_a: Particle, body_b: Particle,
                 ks: float, kd: float = 0., rest_length = 0):
        self.body_a = body_a
        self.body_b = body_b
        self.ks = ks
        self.kd = kd
        self.r = rest_length

    def calc_force(self):
        a = self.body_a.pos
        b = self.body_b.pos
        l = a - b

        a_vel = self.body_a.body_velocity
        b_vel = self.body_b.body_velocity
        l_dot = a_vel - b_vel

        delta_l = np.linalg.norm(l) - self.r
        delta_l_dot = np.dot(l_dot, l) / (np.linalg.norm(l) + 1e-3)

        module = self.ks * delta_l + self.kd * delta_l_dot
        return -module * l / (np.linalg.norm(l) + 1e-3)

    def apply_force(self, bodies):
        hooke_force = self.calc_force()

        self.body_a.force_accumulator += hooke_force
        self.body_b.force_accumulator += -hooke_force

In [ ]:
@dataclass
class PhysicsEngine:
    bodies: List[Particle]
    dynamics_solver: ODESolverBase
    forces: Optional[List[ForceBase]] = field(default_factory=lambda: [])

    def step(self, t0, t1):
        self.zero_forces()
        self.apply_forces()

        s0 = self.get_system_state()
        s1 = self.dynamics_solver.ode(s0, t0, t1, dx_dt_func=self.system_dx_dt_func)
        self.set_system_state(s1)

    def zero_forces(self):
        for body in self.bodies:
            body.zero_forces()

    def apply_forces(self):
        for force in self.forces:
            force.apply_force(self.bodies)

    def get_system_state(self):
        system_state = {}
        for body in self.bodies:
            if len(system_state) == 0:
                system_state = body.get_state()
            else:
                body_state = body.get_state()
                system_state['x'] = np.concatenate((system_state['x'], body_state['x']))
                system_state['p'] = np.concatenate((system_state['p'], body_state['p']))
        return system_state

    def set_system_state(self, new_state: Dict):
        ix_x = 0
        ix_p = 0
        for body in self.bodies:
            body_state = {
                'x': new_state['x'][ix_x: ix_x + body.state_x_size],
                'p': new_state['p'][ix_p: ix_p + body.state_p_size]
            }
            body.set_state(body_state)
            ix_x += body.state_x_size
            ix_p += body.state_p_size

    def system_dx_dt_func(self, system_state):
        ix_x = 0
        ix_p = 0
        system_dx_dt = {'x_dot': [], 'p_dot': []}
        for body in self.bodies:
            body_state = {
                'x': system_state['x'][ix_x: ix_x + body.state_x_size],
                'p': system_state['p'][ix_p: ix_p + body.state_p_size]
            }
            body_dx_dt = body.dx_dt_func(body_state)
            system_dx_dt['x_dot'] = np.concatenate((system_dx_dt['x_dot'], body_dx_dt['x_dot']))
            system_dx_dt['p_dot'] = np.concatenate((system_dx_dt['p_dot'], body_dx_dt['p_dot']))
            ix_x += body.state_x_size
            ix_p += body.state_p_size
        return system_dx_dt

## 1.6 Add visuals

To actually see something in the simulator, we have to synchronize the positions of visual primitives from `VPython` with the particles' positions. For particles, we will use small spheres for visualization.

Let's define an additional structure `Actor` which contains the visual primitive to be rendered and the physics object.

In [ ]:
@dataclass
class Actor:
    name: str 
    phys_obj: Particle
    visual: vp.baseObj

    def __post_init__(self):
        self.update_visuals()
    
    def update_visuals(self):
        self.visual.pos = vp.vector(*self.phys_obj.pos.tolist())

def make_particle_actor(name, color=vp.color.red, radius = 0.2, pos=[0, 0, 0], lin_momentum=[0, 0, 0], mass=1.0):
    return Actor(
        name = name,
        phys_obj = Particle(
            name = name,
            inv_mass = 1 / mass,
            pos = Vec3(pos),
            linear_momentum = Vec3(lin_momentum)
        ),
        visual = vp.sphere(radius=radius, color = color)
    )

The class `SimWorld` will handle both the physics engine's step and the update of visuals.

In [ ]:
class SimWorld:
    def __init__(self, dt, rate, entities = [], **kwargs):
        self.dt = dt
        self.rate = rate
        self.entities = entities
        
        phys_bodies = []
        self.entity_names = []
        self.phys_obj_names = []
        for entity in self.entities:
            if isinstance(entity, Actor):
                phys_bodies.append(entity.phys_obj)
                if entity.phys_obj.name in self.phys_obj_names:
                    raise ValueError(f"Duplicate entity name: {entity.phys_obj.name}")
                self.phys_obj_names.append(entity.phys_obj.name)
            if entity.name in self.entity_names:
                raise ValueError(f"Duplicate entity name: {entity.name}")
            self.entity_names.append(entity.name)

        self.phys_world = PhysicsEngine(
                    bodies = phys_bodies,
                    **kwargs
                )
        
    def update_visuals(self):
        for enity in self.entities:
            enity.update_visuals()

    def step(self, t_0, t_1):
        if len(self.entities) > 0:
            self.phys_world.step(t_0, t_1)
        self.update_visuals()

    def run(self, n_steps = None):
        elapsed_steps = 0
        t = 0
        while True:
            vp.rate(self.rate)
            self.step(t, t + self.dt)
            t += self.dt
            elapsed_steps += 1
            if n_steps is not None and elapsed_steps >= n_steps:
                break

Lastly, let's add visual linkage.

In [ ]:
@dataclass
class VisualLinkageParticles:
    name: str
    actor_1: Actor
    actor_2: Actor
    visual: vp.baseObj
    
    def __post_init__(self):
        self.update_visuals()

    def update_visuals(self):
        start = vp.vector(*self.actor_1.phys_obj.pos.tolist())
        end = vp.vector(*self.actor_2.phys_obj.pos.tolist())
        axis = end - start

        self.visual.pos = start
        self.visual.axis = axis

## 1.7 Simulation Time!

Launch this example in browser:

```
python simple_sim_example_particles.py masses
```

In [ ]:
# masses on a spring
scene = vp.canvas(title='spring')

p1 = make_particle_actor('particle1',
                         pos=[2, 0, 0], lin_momentum=[0, 0, 0], mass=1.0)
p2 = make_particle_actor('particle2',
                         pos=[-2, 0, 0], lin_momentum=[0, 0, 0], mass=1.0)

spring = VisualLinkageParticles('spring',  p1, p2, visual=vp.helix(color = vp.color.green, radius=0.4, thickness=0.1),)

forces = [
    HookeForce(p1.phys_obj, p2.phys_obj, ks=0.8, kd=0.0, rest_length=3.0)
]

dt = 0.01
sim_world = SimWorld(dt, 200, entities = [p1, p2, spring],
                     forces = forces,
                    dynamics_solver=EulerMethod())
sim_world.run(n_steps=2000)

print("FINISHED")

del scene

Launch this example in browser:

```
python simple_sim_example_particles.py hanging
```

In [ ]:
# hanging particle
scene = vp.canvas(title='Particle hanging on the spring')

p1 = make_particle_actor('particle1',
                         pos=[2, 0, 0], lin_momentum=[0, 0, 0], mass=1.0)
p1.phys_obj.inv_mass = 0 # make body with infinite mass


p2 = make_particle_actor('particle2',
                         pos=[-2, 0, 0], lin_momentum=[0, 0, 0], mass=1.0)

spring = VisualLinkageParticles('spring',  p1, p2, visual=vp.helix(color = vp.color.green, radius=0.4, thickness=0.1),)

forces = [
    HookeForce(p1.phys_obj, p2.phys_obj, ks=2.8, kd=1.0, rest_length=3.0),
    GravityForce()
]
dt = 0.01
sim_world = SimWorld(dt, 200, entities = [p1, p2, spring],
                     forces = forces,
                    dynamics_solver=EulerMethod())
sim_world.run(n_steps=2000)

print("FINISHED")
del scene

Launch this example in browser:

```
python simple_sim_example_particles.py wave
```

In [ ]:
# Wave
scene = vp.canvas(title='Wave')

N = 20
l = 2.
    
k_s = 1.5
k_d = 0.11
rest_length = l

center_shift = Vec3((N - 1) * l / 2, 0, 0)

helix_params = dict(color = vp.color.green, radius=0.2, thickness=0.1)
forces = []
entities = []
linkages = []
lin_momentum_scale = 0.1
start_momentum = Vec3(1., 0, 0)

for i in range(-1, N + 1):
    entities.append(make_particle_actor(f'particle_{i}',
                         pos=Vec3(l * i, 0, 0) - center_shift, lin_momentum=[0, 0, 0], mass=1.0, radius=0.5))
    if i == -1 or i == N:
        # entities[-1].visual.visible = False
        entities[-1].phys_obj.inv_mass = 0
        
    if i > -1:
        forces.append(HookeForce(entities[-1].phys_obj, entities[-2].phys_obj, ks=k_s, kd=k_d, rest_length=rest_length))
        linkages.append(VisualLinkageParticles(f'link_{i}', entities[-1], entities[-2], visual=vp.helix(**helix_params)))

    # if i == 1:
entities[1].phys_obj.linear_momentum = start_momentum

entities = entities + linkages

dt=0.05
sim_world = SimWorld(dt, 60, entities = entities,
                     forces = forces,
                    dynamics_solver=EulerMethod())
sim_world.run(n_steps=2000)

print("FINISHED")

del scene

# Part 2. Rigid Bodies

## 2.1 Rigid Bodies Concepts

In contrast to particles, we additionally have orientation of rigid body, which is represented as 3d rotation matrix $\mathbf{R}(t)$, or quaternion $q(t)$. $\mathbf{r}_{COM}$ is the position of the center of mass.

| | Particle | Rigid Body |
| :--- | :---: | :---: |
| Body Properties | mass $m$ | mass $m$, inertia tensor $I(t)=R(t)I_{body}R^T(t)$ |
| Position | position $\mathbf{r}(t)$ | pose $(\mathbf{r}_{COM}(t), R(t))^T$ |
| Velocity | velocity $\mathbf{v}(t)$ | 6d velocity $(\mathbf{v}(t), \mathbf{\omega}(t))^T$ |
| Momentum | momentum $\mathbf{p}(t)$ | linear and angular momenta $(\mathbf{P}(t), \mathbf{L}(t))^T$ |
| Forces | force $\mathbf{F}$ | forces and torques $(\mathbf{F}, \mathbf{\tau})^T$ |

The state $\mathbf{s}$ of the rigid body in the canonical system is specified by a _pose_, i.e. a pair of the center of mass position and rotation quaternion, and a 6D velocity: $$\mathbf{s}(t)=\begin{pmatrix} \mathbf{X}(t) \\ \mathbf{V}(t) \end{pmatrix} = \begin{pmatrix} \mathbf{r}(t) \\ q(t) \\ \mathbf{v}(t) \\ \mathbf{\omega}(t) \end{pmatrix}$$.

__Note__ that $\dot{\mathbf{X}} \neq \mathbf{V}$:

$$\dot{\mathbf{X}}=\begin{pmatrix} \mathbf{\dot{r}}(t) \\ \dot{q}(t) \end{pmatrix}=\begin{pmatrix} \mathbf{v}(t) \\ \frac{1}{2}\omega(t)q(t)\end{pmatrix}=\begin{pmatrix} E_{3\times 3} & 0 \\ 0 & Q \end{pmatrix} \begin{pmatrix} \mathbf{v}(t) \\ \mathbf{\omega}(t) \end{pmatrix} = SV$$

Matrix $S$ - kinematics transformation matrix. And $Q=\frac{1}{2} \begin{pmatrix}\ -x & -y & -z \\ w & z & -y \\ -z & w & x \\ y & -x & w \end{pmatrix}$ for $q=w+xi+yj+zk$


However, it is more convenient to store linear momentum $\mathbf{P}=m\mathbf{v}$ instead of velocity and angular momentum $\mathbf{L}=I\mathbf{\omega}$ instead of angular velocity.

$$\mathbf{s}(t)=\begin{pmatrix} \mathbf{r}(t) \\ q(t) \\ \mathbf{P}(t) \\ \mathbf{L}(t) \end{pmatrix}$$

Right hand side for motion $\dot{\mathbf{s}}=\frac{d}{dt}\mathbf{s}$ equation looks as follows:

$$\frac{d}{dt}\mathbf{s} = \begin{pmatrix} \mathbf{v} \\ \frac{1}{2}\omega(t)q(t) \\ \mathbf{F} \\ \tau \end{pmatrix}$$

This system solves with ODE solver in the same way as for particles.

## 2.2 Rigid Body Structure

We have implemented `Pose` for you, which stores a translation vector and a quaternion:

In [ ]:
pose1 = Pose.from_pq(p=[32, 1, 2], q=[1, 0, 0, 0])
pose1.matrix, pose1.inv().matrix


Now, we are ready to implement `RigidBody`:

In [ ]:
@dataclass
class RigidBody(PhysObject):
    # constants
    inv_mass: float                            # 1 / m
    body_inertia_tensor_inv: np.array          # I_body ^ -1

    # state
    pose: Pose = Pose()                        # x(t), q(t)
    linear_momentum: Vec3 = Vec3(0, 0, 0)      # P(t)
    angular_momentum: Vec3 = Vec3(0, 0, 0)     # L(t)

    # computed quantites
    force_accumulator: Vec3 = Vec3(0, 0, 0)    # F(t) 
    torque_accumulator: Vec3 = Vec3(0, 0, 0)   # tau(t) 

    def zero_forces(self):
        self.force_accumulator = Vec3(0, 0, 0)
        self.torque_accumulator = Vec3(0, 0, 0)
    
    # derived auxiliary variables
    # Note: we make separate functions for each property, which use only 
    # body constants (but no state variables) for calculation 
    # (i.e. `self.body_velocity` and `self.calc_velocity(...)`, 
    # `self.inertia_tensor_inv` and `self.calc_inertia_tensor_inv(...)`, etc.)
    # These calc_* functions are need for `self.dx_dt_func`.
    @property                                  # P(t) / m
    def body_velocity(self):
        return self.calc_velocity(self.linear_momentum)
    
    def calc_velocity(self, linear_momentum):
        return self.inv_mass * linear_momentum
    
    
    @property                                  # R(t)
    def rotation_matrix(self):
        return self.pose.q.rotation_matrix
    
    
    @property
    def inertia_tensor_inv(self):              # I(t) ^ -1
        return self.calc_inertia_tensor_inv(self.pose.q)

    def calc_inertia_tensor_inv(self, q: Quaternion):
        rotation_matrix = q.rotation_matrix
        return rotation_matrix @ self.body_inertia_tensor_inv @ rotation_matrix.T
    

    @property
    def angular_velocity(self):                # omega(t)
        return self.calc_angular_velocity(self.pose.q, self.angular_momentum)
    
    def calc_angular_velocity(self, q: Quaternion, angular_momentum: Vec3):
        inertia_tensor_inv = self.calc_inertia_tensor_inv(self.pose.q)
        return inertia_tensor_inv @ angular_momentum
    
    # return state
    def get_state(self):
        state = dict(
            x = self.pose.raw,
            p = np.concatenate((self.linear_momentum, self.angular_momentum))
        )
        return state

    # set state
    def set_state(self, new_state):
        assert len(new_state['x']) == self.state_x_size
        assert len(new_state['p']) == self.state_p_size
        self.pose = Pose.from_raw(new_state['x'])
        self.linear_momentum = Vec3(new_state['p'][:3])
        self.angular_momentum = Vec3(new_state['p'][3:])

    # 3 for particles, 7 for rigid bodies
    @property
    def state_x_size(self):
        return len(self.get_state()['x'])

    # 3 for particles, 6 for rigid bodies
    @property
    def state_p_size(self):
        return len(self.get_state()['p'])

    @property
    def state_size(self):
        return self.state_x_size + self.state_p_size

    def dx_dt_func(self, x_state):
        x = x_state['x'][:3]
        q = x_state['x'][3:]
        linear_momentum = x_state['p'][:3]
        angular_momentum = x_state['p'][3:]

        body_velocity = self.calc_velocity(linear_momentum)
        angular_velocity = self.calc_angular_velocity(q, angular_momentum)

        q_dot = 0.5 * Quaternion(vector=angular_velocity) * q
        P_dot = self.force_accumulator
        L_dot = self.torque_accumulator

        dx_dt = {}
        dx_dt['x_dot'] = np.concatenate((body_velocity, q_dot.q))
        dx_dt['p_dot'] = np.concatenate((P_dot, L_dot))
        
        return dx_dt
        

Now let's update Hooke's force so it also modifies torques (gravity does not produce torque).

In [ ]:
@dataclass
class HookeForce(ForceBase):
    body_a: Union[Particle, RigidBody]
    body_b: Union[Particle, RigidBody]
    ks: float
    kd: float
    application_point_a: Vec3 = Vec3(0, 0, 0)
    application_point_b: Vec3 = Vec3(0, 0, 0)
    rest_length: float = 1.0

    def calc_force(self):
        com_a_pose = Pose(self.body_a.pos) if isinstance(self.body_a, Particle) else self.body_a.pose
        com_b_pose = Pose(self.body_b.pos) if isinstance(self.body_b, Particle) else self.body_b.pose
    
        a = (com_a_pose * Pose(self.application_point_a)).p
        b = (com_b_pose * Pose(self.application_point_b)).p
        l = a - b

        r_a = a - com_a_pose.p
        r_b = b - com_b_pose.p

        a_vel = self.body_a.body_velocity
        if isinstance(self.body_a, RigidBody):
            a_vel += np.cross(self.body_a.angular_velocity, r_a)
            
        b_vel = self.body_b.body_velocity 
        if isinstance(self.body_b, RigidBody):
            b_vel += np.cross(self.body_b.angular_velocity, r_b)
            
        l_dot = a_vel - b_vel

        delta_l = np.linalg.norm(l) - self.rest_length
        delta_l_dot = np.dot(l_dot, l) / (np.linalg.norm(l) + 1e-3)

        module = self.ks * delta_l + self.kd * delta_l_dot
        return -module * l / (np.linalg.norm(l) + 1e-3)

    def apply_force(self, bodies):
        hooke_force = self.calc_force()

        self.body_a.force_accumulator += hooke_force
        self.body_b.force_accumulator += -hooke_force

        if isinstance(self.body_a, RigidBody):
            r_a = (self.body_a.pose * Pose(self.application_point_a)).p - self.body_a.pose.p
            torque_a = np.cross(r_a, hooke_force)
            self.body_a.torque_accumulator += torque_a

        if isinstance(self.body_b, RigidBody):
            r_b = (self.body_b.pose * Pose(self.application_point_b)).p - self.body_b.pose.p
            torque_b = np.cross(r_b, -hooke_force)
            self.body_b.torque_accumulator += torque_b

That's it for physics. Now let's update `Actor` and visuals, and we are ready.

In [ ]:
@dataclass
class Actor:
    name: str 
    phys_obj: RigidBody
    visual: vp.baseObj

    def __post_init__(self):
        self.update_visuals()
    
    def update_visuals(self):
        if isinstance(self.phys_obj, Particle):
            # particle
            self.visual.pos = vp.vector(*self.phys_obj.pos.tolist())
        else:
            # rigid body
            self.visual.pos = vp.vector(*self.phys_obj.pose.p.tolist())
    
            ox = self.phys_obj.pose.matrix[:3, 0]
            oy = self.phys_obj.pose.matrix[:3, 1]
    
            self.visual.axis = vp.vector(*ox.tolist()) * self.visual.length
            self.visual.up = vp.vector(*oy.tolist()) * self.visual.height


@dataclass
class VisualLinkageBodies:
    name: str
    actor_a: Actor
    application_point_a: Vec3
    actor_b: Actor
    application_point_b: Vec3
    visual: vp.baseObj
    
    def __post_init__(self):
        self.update_visuals()

    def update_visuals(self):
        start = vp.vector(*(self.actor_a.phys_obj.pose * Pose(self.application_point_a)).p.tolist())
        end = vp.vector(*(self.actor_b.phys_obj.pose * Pose(self.application_point_b)).p.tolist())
        axis = end - start

        self.visual.pos = start
        self.visual.axis = axis


# calculate inertia tensor I_body for a box
def get_box_inertia(mass: int, extents: tuple):
    x, y, z = extents
    return mass / 12 * np.array(
        [
            [y ** 2 + z ** 2, 0, 0],
            [0, x ** 2 + z ** 2, 0],
            [0, 0, x ** 2 + y ** 2]
        ]
    )

def make_box_actor(name,
        extents=(1, 1, 1),
        color = vp.color.red,
        pos = [0, 0, 0],
        lin_momentum = [0, 0, 0],
        ang_momentum = [0, 0, 0],
        mass = 1.0
):
    body_inertia_tensor = get_box_inertia(mass, extents)
    return Actor(
        name = name,
        phys_obj=RigidBody(
            name = name,
            inv_mass = 1 / mass,
            body_inertia_tensor_inv = np.linalg.inv(body_inertia_tensor),
            pose = Pose.from_pq(p = pos),
            linear_momentum = Vec3(lin_momentum),
            angular_momentum = Vec3(ang_momentum)
        ), 
        visual=vp.box(color = color, size=vp.vector(*extents), make_trail=False)
    )

## 2.3 Make Simulation not Love!

Launch this example in browser:

```
python simple_sim_example_rigid_bodies.py bodies_spring
```

In [ ]:
# rigid bodies on a spring
scene = vp.canvas()

extents1 = np.array([1, 1, 1])
box1 = make_box_actor('box1', extents1,
                         pos=[3, 0, 0], mass=1.0)
extents2 = np.array([2, 3, 5])
box2 = make_box_actor('box2', extents2,
                         pos=[-3, 0, 0])

spring = VisualLinkageBodies('spring',  box1, extents1 / 2, box2, extents2 / 2,
                       visual=vp.helix(color = vp.color.green, radius=0.4, thickness=0.1),)

forces = [
    HookeForce(box1.phys_obj, box2.phys_obj, application_point_a=extents1 / 2, 
               application_point_b=extents2 / 2,
               ks=0.8, kd=0.0, rest_length=5.0)
]

dt = 0.01
sim_world = SimWorld(dt, 200, entities = [box1, box2, spring],
                     forces = forces,
                    dynamics_solver=EulerMethod())
sim_world.run(n_steps=2000)

print("FINISHED")

Launch this example in browser:

```
python simple_sim_example_rigid_bodies.py hanging_body
```

In [ ]:
# rigid body hanging on a spring
scene = vp.canvas(title='rigid body hanging on a spring')

extents1 = np.array([1, 1, 1])
box1 = make_box_actor('box1', extents1,
                         pos=[3, 0, 0], mass=1.0)
box1.phys_obj.inv_mass = 0
box1.phys_obj.body_inertia_tensor_inv = np.zeros((3, 3))

extents2 = np.array([2, 3, 5])
box2 = make_box_actor('box2', extents2,
                         pos=[-3, 0, 0])

spring = VisualLinkageBodies('spring',  box1, extents1 / 2, box2, extents2 / 2,
                       visual=vp.helix(color = vp.color.green, radius=0.4, thickness=0.1),)

forces = [
    HookeForce(box1.phys_obj, box2.phys_obj, application_point_a=extents1 / 2, 
               application_point_b=extents2 / 2,
               ks=10.8, kd=2.0, rest_length=5.0),
    GravityForce()
]

dt = 0.01
sim_world = SimWorld(dt, 200, entities = [box1, box2, spring],
                     forces = forces,
                    dynamics_solver=EulerMethod())
sim_world.run(n_steps=2000)

print("FINISHED")

Looks plausible.

In [ ]:
# Wave
scene = vp.canvas(title='Wave')

N = 10
l = 2.
    
k_s = 1.5
k_d = 0.2
rest_length = l

center_shift = Vec3((N - 1) * l / 2, 0, 0)

helix_params = dict(color = vp.color.green, radius=0.2, thickness=0.1)
forces = []
entities = []
linkages = []
lin_momentum_scale = 0.1
start_momentum = Vec3(1., 0, 0)

for i in range(-1, N + 1):
    entities.append(make_particle_actor(f'particle_{i}',
                         pos=Vec3(l * i, 0, 0) - center_shift, lin_momentum=[0, 0, 0], mass=1.0, radius=0.5))
    if i == -1 or i == N:
        # entities[-1].visual.visible = False
        entities[-1].phys_obj.inv_mass = 0
        
    if i > -1:
        forces.append(HookeForce(entities[-1].phys_obj, entities[-2].phys_obj, ks=k_s, kd=k_d, rest_length=rest_length))
        linkages.append(VisualLinkageParticles(f'link_{i}', entities[-1], entities[-2], visual=vp.helix(**helix_params)))

    # if i == 1:
entities[1].phys_obj.linear_momentum = start_momentum

entities = entities + linkages

dt=0.05
sim_world = SimWorld(dt, 60, entities = entities,
                     forces = forces,
                    dynamics_solver=EulerMethod())
sim_world.run(n_steps=2000)

print("FINISHED")


# Part 3. Constraint Forces.

## 3.1 Constraints

> The idea of constrained particle dynamics is that our description of the system includes not only
particles and forces, but restrictions on the way the particles are permitted to move. For example,
we might constrain a particle to move along a specified curve, or require two particles to remain
a specified distance apart. The problem of constrained dynamics is to make the particles obey
Newton’s laws, and at the same time obey the geometric constraints.



Constraints are specified by an equation $C(\mathbf{q},\mathbf{\dot{q}}, t)=0$ that the system must satisfy, where $\mathbf{q}$ is the generalized coordinates of the system. A more detailed classification can be seen below.

<img src="constraints.jpeg" width="400" height="300" />

In this seminar, we are focused on __scleronomic__ constraints, which can be represented as $C(\mathbf{q})=0$.

More specifically, let $\mathbf{q}=\begin{pmatrix} \mathbf{r}_1 \\ q_1 \\ ... \\ \mathbf{r}_n \\ q_n \end{pmatrix}$ - poses of all rigid bodies in the system. So $\mathbf{\dot{q}}=S \begin{pmatrix} \mathbf{v}_1 \\ \mathbf{\omega}_1 \\ ... \\ \mathbf{v}_n \\ \mathbf{\omega}_n \end{pmatrix}=SV$, where kinematics matrix $S=diag(S_1, S_2,...,S_N)$ 

And let's define the _mass matrix_ as a $6N\times6N$ block matrix of the following form: 
$$M=\begin{pmatrix} m_1E_{3\times 3} & \mathbf{0} & \mathbf{0} & \mathbf{0} & ... \\ \mathbf{0} & I_1 & \mathbf{0} & \mathbf{0} & ...  \\ \mathbf{0} & \mathbf{0} & m_2E_{3\times 3} & \mathbf{0} & ... \\ \mathbf{0} & \mathbf{0} & \mathbf{0} & I_2 & ... \end{pmatrix}$$

where $m_1$, $m_2$, ..., $m_N$ are the masses of the bodies; $I_1$, $I_2$, ..., $I_N$ are the inertia tensors of the bodies; and $E_{3\times3}$ is a $3\times 3$ identity matrix. 

However, it is more convenient to use matrix $W=M^{-1}$:

$$W=\begin{pmatrix} \frac{1}{m_1}E_{3\times 3} & \mathbf{0} & \mathbf{0} & \mathbf{0} & ... \\ \mathbf{0} & I_1^{-1} & \mathbf{0} & \mathbf{0} & ...  \\ \mathbf{0} & \mathbf{0} & \frac{1}{m_2}E_{3\times 3} & \mathbf{0} & ... \\ \mathbf{0} & \mathbf{0} & \mathbf{0} & I_2^{-1} & ... \end{pmatrix}$$

And let $Q=\begin{pmatrix} \mathbf{F}_1 \\ \mathbf{\tau}_1 \\ ... \\ \mathbf{F}_n \\ \mathbf{\tau}_n \end{pmatrix}$ - forces and torques applied to the bodies.

Then, Newton's 2nd Law can be represented as follows $$M\dot{V}=Q$$ $$\dot{V}=WQ$$

All constraints in the system introduce new forces $Q_c$ on the bodies, which make the bodies follow these constraints. Let's split forces into external forces and constraint forces: 
$$\dot{V}=WQ_{ext}+WQ_c$$

__Our goal here is to find these forces $Q_c$ from $C(\mathbf{q})=0$.__

One important concept worth mentioning here is called the __principle of virtual work.__ According to this principle, the work of constraint forces always equals 0: $$Q_c^Td\mathbf{q}=0$$

if we divide this by $dt$, we get $$Q_c^TV=0$$

To understand the principle of virtual work, you can imagine a particle moving along a rigid rod (or any other rigid curve). It is obvious that all forces acting on the particle from the side of the rod will be directed perpendicular to the particle's velocity.

## 3.2 Constraint Derivations. Jacobian

Let's differentiate $C(\mathbf{q})=0$:

$$\dot{C}(\mathbf{q})=\frac{\partial C}{\partial \mathbf{q}} \dot{\mathbf{q}}=\frac{\partial C}{\partial \mathbf{q}} S V$$

matrix $J=\frac{\partial C}{\partial \mathbf{q}}S$ is called a Jacobian of $C$.

$$\dot{C}=JV=0$$

let's differentiate once again:

$$\ddot{C}=\dot{J}V+J\dot{V}=0$$

The matrix $\dot{J}=\frac{\partial J}{\partial \mathbf{q}}\dot{\mathbf{q}}$ leads to a 4-dimensional tensor $\frac{\partial J}{\partial \mathbf{q}}$, which is difficult for further derivations; however, we have a workaround: $\dot{J}=\frac{\partial \dot{C}}{\partial \mathbf{q}}$S.

Let's substitute here $\dot{V}=WQ_{ext}+WQ_c$. We get:

$$\dot{J}V+JWQ_{ext}+JWQ_c=0$$

$$JWQ_c=-\dot{J}V-JWQ_{ext}$$

The condition $JV=0$ gives the equation for all legal velocities. Combined with the principle of virtual work, it gives:

$$Q_c^TV=0, \forall V | JV=0$$

All and only vectors $Q_c$ that satisfy this requirement can be expressed in the form:

$$Q_c=J^T\lambda$$

where $\lambda$ is a vector with the dimension of $C$ (Lagrange multipliers).

Let's substitute it to equation:

$$JWJ^T\lambda=-\dot{J}V-JWQ_{ext}$$

$$\lambda=(JWJ^T)^{-1}(-\dot{J}V-JWQ_{ext})$$

For numerical stability additional (feedback) terms are added:

$$\lambda=(JWJ^T)^{-1}(-\dot{J}V-JWQ_{ext}-k_sC-k_d\dot{C})$$

## 3.3 Example: Fixed Point of the Rigid Body

Imagine the body is fixed by a point to a hinge around which it can rotate freely.

<img src="fixed_point.jpeg" width="400" height="300" />

The center of mass is given by $\mathbf{r}$, and the point of fixation on the body is given by $\mathbf{r}_{f\_local}$ in the body frame. The hinge point is given by the vector $\mathbf{R}$ in the world frame. At each time step, we transform this vector $\mathbf{r}_{f\_local}$ to the world frame: $\mathbf{r}_f = R(t)\mathbf{r}_{f\_local}$. The constraint can then be written as:

$$C(\mathbf{r})=\mathbf{r}+\mathbf{r}_f-\mathbf{R}=\mathbf{0}$$

Differentiate it:

$$\dot{C}=\mathbf{v}+\omega \times \mathbf{r}_f=JV$$

Since generalized velocity $V$ equals $(\mathbf{v}, \mathbf{\omega})^T$ and fact that $\omega \times \mathbf{r}_f=-\mathbf{r}_f \times \omega = -[\mathbf{r}_f]_{\times}\omega$ we get:

$$\dot{C}=\begin{pmatrix}E_{3\times 3} & -[\mathbf{r}_f]_{\times} \end{pmatrix} \begin{pmatrix} \mathbf{v} \\ \mathbf{\omega} \end{pmatrix}=JV$$

We obtain our $3\times 6$ Jacobian matrix:

$$J=\begin{pmatrix}E_{3\times 3} & -[\mathbf{r}_f]_{\times} \end{pmatrix}$$

where $E_{3\times 3}$ is an identity matrix.

Next we can directly compute $\dot{J}=\begin{pmatrix} 0 & -[\mathbf{\dot{r}}_f]_{\times} \end{pmatrix}=\begin{pmatrix} 0 & -[\mathbf{\omega}\times\mathbf{r}_f]_{\times} \end{pmatrix}$

Then, $\dot{J}V=-[\mathbf{\omega}\times\mathbf{r}_f]_{\times} \mathbf{\omega} = \mathbf{\omega} \times (\mathbf{\omega} \times \mathbf{r}_f)$


## 3.4 Implementing Constrained Dynamics

The sequence of steps that the system must perform to evaluate the derivative is nearly the same, with one important addition:
calculating the constraint force. This is how the extra step fits in:

1. Clear forces. Zero each body's force and torque accumulators.
2. Calculate forces: loop over all force objects, allowing each to add forces to the bodies it influences.
3. Calculate constraint forces: On completion of the previous step, each particle’s force accumulator contains the total force on that particle. In this step, the global equation 11 is set up and solved, yielding a constraint force on each particle, which is added into the applied force.
4. ODE solution.

Let's focus on the 3rd step here. There are two goals:

1. We want modular design for constraints as we have for forces.
2. For large systems with many constraints and bodies, matrices $J$, $\dot{J}$, $W$ become very large.

Let's look at matrix $J$. For example, if have two constraints $C_1$ and $C_2$, where the first one applies constraint to n-th and m-th bodies, and the second applies to k-th and m-th.
Then $J$ looks as follows:

$$J=\begin{pmatrix} ... &  0 & 0 & \frac{\partial C_1}{\partial \mathbf{q}_n} & 0 & ... & 0 & \frac{\partial C_1}{\partial \mathbf{q}_m} \\ ... &  0 & \frac{\partial C_2}{\partial \mathbf{q}_k} & 0 & 0 & ... & 0 & \frac{\partial C_2}{\partial \mathbf{q}_m} \end{pmatrix}$$

where each block has size $3\times 6$ if it is a rigid body constraint, and $1\times 3$ if it is a particle constraint. Therefore, for each constraint we can store which blocks in $J$ are modified by this particular constraint using this structure:

```
@dataclass
class MatrixCell:
    i_start: int = None
    j_start: int = None
    i_end: int = None
    j_end: int = None
```

To handle the large sizes of $J$ and $W$, we can use different sparse representations for matrices, such as CSR or CSC. This effectively reduces memory consumption, especially for a large number of bodies/particles.

To solve the equation for $\lambda$, we can use the conjugate gradients method, as it effectively applies to sparse matrices (requiring only matrix-vector multiplications).

To sum up, we will have one instance of `ConstraintsManager` that handles updates of sparse matrices $J$, $\dot{J}$ from one or many `Constraint`s.

## 3.5 Code It

First, let's define the constraint interface. Note that the implementation may vary, but here we will not store $\dot{J}$; instead, every constraint will calculate $\dot{J}V$ and pass it to `ConstraintsManager`.

In [ ]:
class Constraint(ABC):
    @abstractmethod
    def update(self, t_0, t_1):    # update internal values
        pass

    @property
    @abstractmethod
    def constrained_bodies(self):  # return names of the bodies under constraint
        pass

    @property
    @abstractmethod                # number of rows in C
    def dim_size(self):
        pass

    @abstractmethod                # calculate C(r)
    def get_C(self):
        pass

    @abstractmethod
    def get_J_updates(self):       # calculate dJ/dq_i for each body under constraint
        pass
    

    @abstractmethod                # calculate  J' V
    def get_J_dot_V(self):
        pass


In [ ]:
@dataclass
class MatrixCell:
    i_start: int = None
    j_start: int = None
    i_end: int = None
    j_end: int = None

    @property
    def slice_shape(self):
        i_size = 0
        if not (self.i_start is None or self.i_end is None):
            i_size = self.i_end - self.i_start
        j_size = 0
        if not (self.j_start is None or self.j_end is None):
            j_size = self.j_end - self.j_start
        return (i_size, j_size)

In [ ]:
from scipy.sparse import csc_array
from scipy.sparse.linalg import cg

@dataclass
class ConstraintsManager:
    bodies: List[RigidBody]
    constraints: Dict[str, Constraint]
    k_s: float = 10000
    k_d: float = 100
    
    def __post_init__(self):
        self.J, self.J_idxs = self._construct_empty_jacobian()
        self.J_num_rows, self.J_num_cols = self.J.shape

        self.W, self.W_idxs = self._construct_system_inv_inertia()


    def _construct_empty_jacobian(self):
        row = 0
        col = 0
        J_idxs = {} # dict of dicts; J_idxs[constraint_name][body] -> idxs in J

        for body in self.bodies:
            row = 0

            for constraint_name, constraint in self.constraints.items():
                if not constraint_name in J_idxs:
                    J_idxs[constraint_name] = {}    

                J_idxs[constraint_name][body.name] = MatrixCell(
                    i_start = row, j_start = col,
                    i_end = row + constraint.dim_size,
                    j_end = col + body.state_p_size
                )
                row += constraint.dim_size

            col += body.state_p_size

        return csc_array((row, col), dtype=np.float32), J_idxs

    def _construct_system_inv_inertia(self):
        W = csc_array((self.J_num_cols, self.J_num_cols), dtype=np.float32)
        W_idxs = {}
        i, j = 0, 0
        for body in self.bodies:
            idx = MatrixCell(i, j, i + body.state_p_size, j + body.state_p_size)
            W_idxs[body.name] = idx

            # fill inv masses since they are constant
            W[idx.i_start : idx.i_start + 3, idx.j_start : idx.j_start + 3] = body.inv_mass * np.eye(3)
            i += body.state_p_size
            j += body.state_p_size
        return W, W_idxs

    def step(self, t_0, t_1):
        if len(self.constraints) == 0:
            return
        # calculate internal values in constraints
        self.update_constraints(t_0, t_1)

        # get sparse matrices values

        self.update_W()
        W = self.W # system invese inertia matrix
        
        self.update_J()
        J = self.J # constrains jacobian

        # get vectors
        J_dot_V = self.get_J_dot_V() # J'V
        C = self.get_C() # constrains values
        V = self.get_V() # system velocity 6N vector (for particles 3N)
        F = self.get_F() # system force 6N vector (for particles 3N)

        # calculate K=JWJ^T
        K = J @ W @ J.T

        rhs = -J_dot_V - J @ W @ F - self.k_d * J @ V - self.k_s * C
        # solve using conjugade gradients
        l, exit_code = cg(K, rhs, atol=1e-5)
        assert exit_code == 0

        F_C = J.T @ l

        # update forces (and torques)
        self.update_forces(F_C)
    
    def update_W(self):
        for body in self.bodies:
            idx = self.W_idxs[body.name]
            if idx.slice_shape[1] == 3: # skip particle update
                continue
            self.W[idx.i_start + 3 : idx.i_end,
                   idx.j_start + 3 : idx.j_end] = body.inertia_tensor_inv
            
    def update_J(self):
        for constraint_name, constraint in self.constraints.items():
            updates = constraint.get_J_updates()
            for body_name, J_update in updates.items():
                idx = self.J_idxs[constraint_name][body_name]
                assert idx.slice_shape == J_update.shape
                self.J[idx.i_start : idx.i_end,
                   idx.j_start : idx.j_end] = J_update
                
    def get_C(self):
        C = []
        for _, constraint in self.constraints.items():
            C.append(constraint.get_C())
        C = np.concatenate(C)
        assert C.shape[0] == self.J_num_rows
        return C
    
    def get_J_dot_V(self):
        J_dot_V = []
        for _, constraint in self.constraints.items():
            J_dot_V.append(constraint.get_J_dot_V())
        J_dot_V = np.concatenate(J_dot_V)
        assert J_dot_V.shape[0] == self.J_num_rows
        return J_dot_V

    def get_V(self):
        V = []
        for body in self.bodies:
            V.append(body.body_velocity)
            if body.state_p_size == 6:
                V.append(body.angular_velocity)
        V = np.concatenate(V)
        assert V.shape[0] == self.J_num_cols
        return V
    
    def get_F(self):
        F = []
        for body in self.bodies:
            F.append(body.force_accumulator)
            if body.state_p_size == 6:
                F.append(body.torque_accumulator)
        F = np.concatenate(F)
        assert F.shape[0] == self.J_num_cols
        return F

    def update_constraints(self, t_0, t_1):
        for _, constraint in self.constraints.items():
            constraint.update(t_0, t_1)


    def update_forces(self, F_C):
        i = 0
        for body in self.bodies:
            f_c = F_C[i : i + 3]
            body.force_accumulator += f_c

            i += 3

            if body.state_p_size == 6: # rigid body
                tau_c = F_C[i : i + 3]
                body.torque_accumulator += tau_c

                i += 3

Let's add the hinge constraint from the example:

In [ ]:
@dataclass
class BallAndSocketPoint(Constraint):
    '''
    Body point with coordinates r_f_local in the body frame
    is fixed in space. The body is free to rotate around this fixed point.

    r is the COM position of the body. 
    At each timestep we transform r_f_local to r_f in the world frame.
    Initially, R := r_f at time t = 0 and is constant during the whole simulation.

    So the constraint uquation looks as follows:

    C(r) = r + r_f - R

    Args
    ----
    body: RigidBody
        Body under constraint.

    body_fixed_point_local: Vec3
        Body anchor point in body frame (r_f_local).
    '''
    body: RigidBody
    body_fixed_point_local: Vec3 # r_f_local

    def __post_init__(self,): # Calc R
        self.fixed_point_world = (self.body.pose * Pose(p=self.body_fixed_point_local)).p

    def update(self, t_0, t_1):
        '''
        find r_f from r_f_local
        '''
        self.body_fixed_point_world = self.body.pose.q.rotation_matrix @ self.body_fixed_point_local

    @property
    def constrained_bodies(self):
        return [self.body]

    @property
    def dim_size(self):
        return 3

    def get_C(self):
        '''
        C = r + r_f - R
        '''
        return self.body.pose.p + self.body_fixed_point_world - self.fixed_point_world 

    def get_J_updates(self):
        '''
        J = [ E | - [r_f]x ]
        '''
        r_f_star = vec_star(self.body_fixed_point_world)
        return {
            self.body.name: np.hstack((np.eye(3), -r_f_star))
        }
    
    def get_J_dot_V(self):
        '''
        J'V = omega x (omega x r_f)
        '''
        return np.cross(self.body.angular_velocity, np.cross(self.body.angular_velocity, self.body_fixed_point_world))

## 3.6 Simulation

We have to add constraints and the manager to `PhysicsEngine`:

In [ ]:
@dataclass
class PhysicsEngine:
    bodies: List[Particle]
    dynamics_solver: ODESolverBase
    forces: Optional[List[ForceBase]] = field(default_factory=lambda: [])
    constraints_manager: Optional[ConstraintsManager] = None

    def apply_constraints(self, t0, t1):
        self.constraints_manager.step(t0, t1)

    def step(self, t0, t1):
        self.zero_forces()
        self.apply_forces()
        self.apply_constraints(t0, t1)

        s0 = self.get_system_state()
        s1 = self.dynamics_solver.ode(s0, t0, t1, dx_dt_func=self.system_dx_dt_func)
        self.set_system_state(s1)

    def zero_forces(self):
        for body in self.bodies:
            body.zero_forces()

    def apply_forces(self):
        for force in self.forces:
            force.apply_force(self.bodies)

    def get_system_state(self):
        system_state = {}
        for body in self.bodies:
            if len(system_state) == 0:
                system_state = body.get_state()
            else:
                body_state = body.get_state()
                system_state['x'] = np.concatenate((system_state['x'], body_state['x']))
                system_state['p'] = np.concatenate((system_state['p'], body_state['p']))
        return system_state

    def set_system_state(self, new_state: Dict):
        ix_x = 0
        ix_p = 0
        for body in self.bodies:
            body_state = {
                'x': new_state['x'][ix_x: ix_x + body.state_x_size],
                'p': new_state['p'][ix_p: ix_p + body.state_p_size]
            }
            body.set_state(body_state)
            ix_x += body.state_x_size
            ix_p += body.state_p_size

    def system_dx_dt_func(self, system_state):
        ix_x = 0
        ix_p = 0
        system_dx_dt = {'x_dot': [], 'p_dot': []}
        for body in self.bodies:
            body_state = {
                'x': system_state['x'][ix_x: ix_x + body.state_x_size],
                'p': system_state['p'][ix_p: ix_p + body.state_p_size]
            }
            body_dx_dt = body.dx_dt_func(body_state)
            system_dx_dt['x_dot'] = np.concatenate((system_dx_dt['x_dot'], body_dx_dt['x_dot']))
            system_dx_dt['p_dot'] = np.concatenate((system_dx_dt['p_dot'], body_dx_dt['p_dot']))
            ix_x += body.state_x_size
            ix_p += body.state_p_size
        return system_dx_dt

Launch this example in browser:

```
python python simple_sim_example_constraints.py
```

In [ ]:
scene = vp.canvas(title='Fixed hinge')

extents1 = np.array([2, 3, 15])
box1 = make_box_actor('box1', extents1, color=vp.color.green,
                         pos=[0, 0, 0], mass=1.0)

constraints = {
    "fixed_point1": BallAndSocketPoint(
        body=box1.phys_obj,
        body_fixed_point_local=Vec3(extents1 / 2), 
    ),
}

forces = [
    GravityForce()
]

fixed_point = vp.sphere(pos=vp.vector(*(extents1 / 2).tolist()), radius=0.5, color = vp.color.red)

dt = 0.01
sim_world = SimWorld(dt, 200, entities = [box1],
                     forces = forces,
                    dynamics_solver=EulerMethod(),
                    constraints_manager=ConstraintsManager(bodies = [box1.phys_obj], constraints=constraints))
sim_world.run(n_steps=2000)

print("FINISHED")

# Part 4. Contact Forces

Last but not least is the topic of __contact forces__, i.e. forces acting only when the geometrical shapes of bodies are intersecting (or __colliding__). Roughly speaking, there are two main stages in simulating contact forces:

1. __Collision detection__. We test each pair of bodies if there is a collision between them.
2. __Collision response__. Consists of two parts usually:

    2a. Force (or impulse) computation for each body to prevent interpenetration. This corresponds to calculation of normal $f_n$ component of the contact force.
   
    2b. Friction force computation. This corresponds to calculation of tangential component of the contact force.

## 4.1 Collision Detection

### 4.1.1 Narrow and Broad Phases

Collision detection is the problem of determining "if" and "where" two bodies are in contact. “If” involves establishing a Boolean
result, answering the question whether or not the objects intersect. “Where” establishes how the objects are coming into contact. Because any one object can potentially collide with any other object, a simulation with $N$ objects requires $O(N^2)$ pairwise tests. This is reduced by separating the collision test into a **broad phase** and a **narrow phase**:

* The broad phase identifies smaller groups of objects that may be colliding and quickly excludes those that definitely are not.
* The narrow phase constitutes the pairwise tests within subgroups.

<img src="broad_vs_narrow.jpeg" width="600" height="450" />

For more details [see](http://www.r-5.org/files/books/computers/algo-list/realtime-3d/Christer_Ericson-Real-Time_Collision_Detection-EN.pdf).

### 4.1.2 Pairwise Collision Detection

The most interesting part for us is a pairwise detection. For two convex polyhedra we aim to find separating plane between them.  If a pair of convex polyhedra are disjoint or contacting (but not inter-penetrating), then a separating plane exists with the following property: **either the plane contains a face of one of the polyhedra, or the plane contains an edge from one of the polyhedra and is parallel to an edge of the other polyhedra.** 

<img src="sep_planes.jpeg" width="600" height="450" />

If no separating plane can be found, then the two polyhedra must be inter-penetrating. When two polyhedra inter-penetrate, it is almost always the case that either a vertex of one polyhedron is inside the other, or an edge of one polyhedron has intersected a face of the other.

Altogether, the output of the collision detector will be a struct containing all collision information:


```
@dataclass
class Collision:
    a: RigidBody  # body containing vertex
    b: RigidBody  # body containing face 
    p: Vec3       # world-space vertex location
    n: Vec3       # outwards pointing normal of face
    ea: Vec3      # edge direction for A 
    eb: Vec3      # edge direction for B 
    vf: bool      # True if vertex/face contact

...

cd = CollisionDetector(bodies)
cd.detect() -> List[Collision]
```

* For a vertex/face contact, `a` is the body with the vertex, `b` is the body with the face (we call these bodies A and B, respectively), `ea` and `eb` are unused. `p` is the position of the penetrating vertex.
* For edge/edge contacts, `ea` - vector collinear to edge in A, `eb` - vector collinear to edge in B. `n` denotes `ea x eb` direction. `p` is where two edges intersect.


## 4.2 Collision response

When collisions are found, we have to handle them to avoid interpenetration. There are several approaches to collision response.

### 4.2.1 Penalty Method

This is generally done by applying spring calculations to the colliding objects - the more the objects penetrate, the more the ’spring’ that connects them wants to snap back to its resting length.

<img src="penalty.jpeg" width="600" height="450" />

$$F=-k_s\Delta x$$

where $\Delta x$ is an interpenetration depth and $k_s$ is large enough coefficient.

Although this method is easy to implement, it requires a sufficiently small $\Delta t$ for integration, and bodies with large masses may still slightly interpenetrate.

### 4.2.2 Impulse Method



In contrast to the penalty method, where we modify `force_accumulator` and `torque_accumulator`, the impulse-based method directly modifies the linear and angular momenta of colliding objects. The change of linear moment is calculated:

<img src="impulse.jpeg" width="600" height="450" />

This technique is also relatively simple, prevents small interpenetrations during simulation, and does not require a small $\Delta t$.

### 4.2.3 Contact as a Constraint

Let $\mathbf{p}_a$ and $\mathbf{p}_b$ be contact points on A and B. If $\mathbf{x}_a$ and $\mathbf{x}_b$ are positions of COMs, and $\mathbf{r}_a$, $\mathbf{r}_b$ are vectors from the COM of each body to the contact points, we have:

$$\mathbf{p}_a=\mathbf{x}_a+\mathbf{r}_a$$ 
$$\mathbf{p}_b = \mathbf{x}_b + \mathbf{r}_b$$

The surface normal $\mathbf{n}_a$ at the contact point $\mathbf{p}_a$.

Then penetration constraint function $C_{pen}=(\mathbf{p}_b-\mathbf{p}_a)\mathbf{n}_a=(\mathbf{x}_b + \mathbf{r}_b - \mathbf{x}_a-\mathbf{r}_a)\mathbf{n}_a \geq 0$

If we compute time derivative:

$$\dot{C} \approx \begin{pmatrix} -\mathbf{n}_a^T & -(\mathbf{r}_a \times \mathbf{n}_a)^T & \mathbf{n}_a^T & (\mathbf{r}_b \times \mathbf{n}_a)^T \end{pmatrix} \begin{pmatrix} \mathbf{v}_1 \\ \mathbf{\omega}_1 \\ \mathbf{v}_2 \\ \mathbf{\omega}_2 \end{pmatrix}$$

then the Jacobian matrix is:

$$J=\begin{pmatrix} -\mathbf{n}_a^T & -(\mathbf{r}_a \times \mathbf{n}_a)^T & \mathbf{n}_a^T & (\mathbf{r}_b \times \mathbf{n}_a)^T \end{pmatrix}$$

from which we can obtain normal contact force $f_n$ as earlier.

## 4.3 Friction

### 4.3.1 Coulomb Law

Friction is a bit trickier. If the tangential component of the relative velocity of contact points is zero (bodies stick to each other), then $f_t$ equals **any value** necessary to maintain $\mathbf{v}_{rel}=0$. If bodies move in the tangential direction, then the friction force obeys Coulomb's Law:

$$f_t=-\mu f_n\frac{\mathbf{v}_{rel}}{||\mathbf{v}_{rel}||}$$

In other words, the friction force lies inside the **friction cone**:

$$||f_t||\leq \mu ||f_n||$$

<img src="cone.jpeg" width="600" height="450" />

### 4.3.2 Friction as a Constraint

Let us take two orthonormal directions $\mathbf{t}_1$, $\mathbf{t}_2$ in the tangential plane. Our friction force can be expressed as $f_t=\lambda_{t1}\mathbf{t}_1+\lambda_{t2}\mathbf{t}_2$.

For each friction component, we can write constraints to slow down the rigid bodies in the direction of the two vectors $\mathbf{t}_1$, $\mathbf{t}_2$. Jacobian matrices for these two constraints:

$$J_{t1}=\begin{pmatrix} -\mathbf{t}_1^T & -(\mathbf{r}_a \times \mathbf{t}_1)^T & \mathbf{t}_1^T & (\mathbf{r}_b \times \mathbf{t}_1)^T \end{pmatrix}$$

$$J_{t2}=\begin{pmatrix} -\mathbf{t}_2^T & -(\mathbf{r}_a \times \mathbf{t}_2)^T & \mathbf{t}_2^T & (\mathbf{r}_b \times \mathbf{t}_2)^T \end{pmatrix}$$

Our total friction force equals:

$$f_t=J_{t1}^T\lambda_{t1}+J_{t2}^T\lambda_{t2}$$

From Coloumb constraint we get:

$$\lambda_{t1}^2+\lambda_{t2}^2 \leq \mu^2 \lambda_{n} ^ 2$$

Finding $\lambda_{t1}$, $\lambda_{t2}$ is formulated as a **non-linear complementarity problem (NCP)**.

However, we can represent it as a linear problem.

### 4.3.3 Cone Approximation

We can approximate our cone with a **pyramid**:

<img src="cone_approx.jpeg" width="600" height="450" />

$$-\mu \lambda_n \leq \lambda_{t1} \leq \mu \lambda_n$$

$$-\mu \lambda_n \leq \lambda_{t2} \leq \mu \lambda_n$$


And we can clamp our components $\lambda_{t1}$, $\lambda_{t2}$:

$$\lambda_{ti} \leftarrow clamp(\lambda_{ti}, -\mu \lambda_n, \mu \lambda_n)$$
